# Exercise 8.02 - Creating a Choropleth Plot with GeoJSON

**Course:** CISC7204 Suggested Exercise 08  
**Practice:** Practice 02

This notebook follows the requested workflow step by step and keeps the
supplied data file in the same `Jupyter Notebooks` folder.

> **Environment note:** These visualizations open in a separate interactive
> window. `geoplotlib 0.3.2` is a legacy package and does not install directly
> into the current Python 3.14 environment. Use a Python 3.10 or 3.11
> Anaconda environment for this tutorial. In **Anaconda Prompt**, run:
>
> ```text
> conda create -n cisc7204-py311 python=3.11 -y
> conda activate cisc7204-py311
> python -m pip install --no-build-isolation "setuptools<58" "geoplotlib==0.3.2" "pyglet<2"
> ```
>
> Then select the `cisc7204-py311` kernel in JupyterLab, restart the kernel,
> and run the notebook from the top. Close each map window before running the
> next visualization cell.

## 1. Load and inspect the GeoJSON data

In [1]:
from pathlib import Path

DATA_FILE = Path("us_states.json")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} was not found. Put this notebook in the same "
        "'Jupyter Notebooks' folder as the supplied data file."
    )
print("Using:", DATA_FILE.resolve())

Using: ../data/us_states.json


In [2]:
import copy
import json
import geoplotlib
from geoplotlib.colors import ColorMap
from geoplotlib.utils import BoundingBox

with DATA_FILE.open(encoding="utf-8") as data:
    state_geojson = json.load(data)

fourth_state = copy.deepcopy(state_geojson["features"][3])
coordinates = fourth_state["geometry"]["coordinates"]
while isinstance(coordinates, list) and coordinates and isinstance(coordinates[0], list):
    coordinates = coordinates[0]
fourth_state["geometry"]["coordinates"] = coordinates
print(json.dumps(fourth_state, indent=4))

{
    "type": "Feature",
    "properties": {
        "GEO_ID": "0400000US05",
        "STATE": "05",
        "NAME": "Arkansas",
        "LSAD": "",
        "CENSUSAREA": 52035.477
    },
    "geometry": {
        "type": "Polygon",
        "coordinates": [
            -94.042964,
            33.019219
        ]
    }
}


In [3]:
def use_arcgis_tiles():
    geoplotlib.tiles_provider({
        "url": lambda zoom, xtile, ytile:
            "https://server.arcgisonline.com/ArcGIS/rest/services/"
            "World_Street_Map/MapServer/tile/%d/%d/%d.jpg"
            % (zoom, ytile, xtile),
        "tiles_dir": "arcgis_world_street",
        "attribution": "Esri World Street Map",
    })

In [4]:
states = [feature["properties"]["NAME"] for feature in state_geojson["features"]]
print("Number of state features:", len(states))
print(states)

Number of state features: 52
['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming', 'Puerto Rico']


## 2. State outlines

In [5]:
use_arcgis_tiles()
geoplotlib.geojson(str(DATA_FILE))
geoplotlib.set_bbox(BoundingBox.USA)
geoplotlib.show()

## 3. Census-area color mapping

In [6]:
cmap = ColorMap("Reds", alpha=255, levels=10)

def get_color(properties):
    return cmap.to_color(
        properties["CENSUSAREA"], maxvalue=300000, scale="lin"
    )

In [7]:
use_arcgis_tiles()
geoplotlib.geojson(str(DATA_FILE), fill=True, color=get_color)
geoplotlib.geojson(str(DATA_FILE), fill=False, color=[0, 0, 0, 255])
geoplotlib.set_bbox(BoundingBox.USA)
geoplotlib.show()

## 4. Interactive tooltip

In [8]:
use_arcgis_tiles()
def state_tooltip(properties):
    return (
        f'{properties["NAME"]} - Census Area: '
        f'{properties["CENSUSAREA"]:,.3f}'
    )

geoplotlib.geojson(
    str(DATA_FILE), fill=True, color=get_color, f_tooltip=state_tooltip
)
geoplotlib.geojson(str(DATA_FILE), fill=False, color=[0, 0, 0, 255])
geoplotlib.set_bbox(BoundingBox.USA)
geoplotlib.show()

Darker red indicates a larger census area. Hovering over a state displays its name and census-area value.